In [1]:
import pandas as pd
from pathlib import Path

# Dataset: Kaggle "IEEE-CIS Fraud Detection" competition data. ~1.3GB and
# gated behind competition acceptance, so it's not committed to this repo -
# download it yourself and point DATA_DIR at it (defaults to the shared
# local path used during development of this project).
DATA_DIR = Path("../../data/ieee-fraud-detection")

# Step 1: load transaction + identity tables, left-join on TransactionID.
# Only ~24% of transactions have a matching identity record - the rest are
# NaN on every identity field, which is itself informative, not missing
# data to impute around.
tx = pd.read_csv(DATA_DIR / "train_transaction.csv")
idn = pd.read_csv(DATA_DIR / "train_identity.csv")
df = tx.merge(idn, on="TransactionID", how="left")

# Step 2: sort by TransactionDT for a time-based split (next cell), not a
# random one - fraud patterns drift over time, and a random split leaks
# near-future information through correlated card/device activity,
# overstating real-world performance.
df = df.sort_values("TransactionDT").reset_index(drop=True)
print(df.shape, "fraud rate:", df["isFraud"].mean())

(590540, 434) fraud rate: 0.03499000914417313


In [2]:
from sklearn.preprocessing import OrdinalEncoder

ID_COLUMNS = ("TransactionID", "isFraud", "TransactionDT")
feature_cols = [c for c in df.columns if c not in ID_COLUMNS]

split_idx = int(len(df) * 0.8)
X_train = df[feature_cols].iloc[:split_idx].copy()
X_test = df[feature_cols].iloc[split_idx:].copy()
y_train = df["isFraud"].iloc[:split_idx]
y_test = df["isFraud"].iloc[split_idx:]

# Categorical columns are determined empirically (string dtype after the
# merge), not guessed from Kaggle's documentation, since the actual CSV
# dtypes are the ground truth that has to match at serving time too.
categorical_columns = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

# One encoder, fit on the training split only, then applied - never
# re-fit - to the test split and to every live request. Plays the same
# "fit once, reapply identically" role StandardScaler played when this
# notebook trained on the old creditcard.csv dataset. Numeric columns are
# NOT scaled this time: LightGBM is tree-based and scale-invariant, so a
# scaler here would be dead weight.
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_columns] = encoder.fit_transform(X_train[categorical_columns].astype(str)).astype("float64")
X_test[categorical_columns] = encoder.transform(X_test[categorical_columns].astype(str)).astype("float64")
print(len(feature_cols), "features,", len(categorical_columns), "categorical")

431 features, 31 categorical


In [3]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

# Landed here via a sweep (same as product/sentryscore/training/train.py):
# scale_pos_weight=1 -> 0.5605, =3 w/ more trees -> 0.581, this config ->
# 0.5926 PR-AUC on the time-based validation split. Note this isn't
# comparable to the old creditcard.csv model's 0.8446 - different
# dataset, and a stricter time-based split here instead of a random one.
model = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=127,
    min_child_samples=50,
    feature_fraction=0.7,
    bagging_fraction=0.8,
    bagging_freq=1,
    scale_pos_weight=2,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"PR-AUC Score: {pr_auc:.4f}")

# Threshold derived from this model's own precision/recall curve, not
# copied over from the old dataset's unrelated 0.75.
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
threshold = float(thresholds[f1[:-1].argmax()])
print(f"F1-optimal threshold: {threshold:.4f} (F1={f1_score(y_test, y_pred_proba > threshold):.4f})")

[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.493167 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34278
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


PR-AUC Score: 0.5926
F1-optimal threshold: 0.2891 (F1=0.5734)


Result: PR-AUC Score 0.5926, F1-optimal threshold 0.2891 (matches `product/sentryscore`'s v2.0.0 model exactly - same dataset, same methodology). Lower than the old creditcard.csv model's 0.8446, but not comparable: this evaluation uses a stricter time-based split (see the comment above `OrdinalEncoder`) instead of a random one, and this dataset's fraud patterns are genuinely harder to separate without heavier feature engineering than top public competition solutions use (out of scope here).

In [4]:
import joblib

# Save the model and categorical encoder (no scaler this time - see the
# comment above OrdinalEncoder for why).
joblib.dump(model, "fraud_model.joblib")
joblib.dump(encoder, "encoder.joblib")
print("Model and encoder saved successfully.")

Model and encoder saved successfully.
